<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Change Index Processor Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load many entities
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 - Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug functions from processor_change_index_functions.py**

### 🔍 Configure extraction

`setup_change_index_parameters` controls the satellite collections, vegetation index, and the look-back windows used to pick the previous image. The **reference image date** is supplied **per entity** via the `reference_date` column (column_mapping-aware) — no coverage lookup happens inside the processor.

In [ ]:
from earthdaily.agriculture.processors.processor_change_index_functions import ChangeIndexExtractor

extractor = ChangeIndexExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)

extractor.setup_change_index_parameters(
    map_type="NDVI",                 # NDVI, EVI, CVI, GNDVI, NDWI
    collections=["Sentinel-2"],      # Sentinel-2, Landsat
    max_period_reference=7,           # Max days back from reference_date to find the reference image
    max_period_previous=15,           # Max days before the reference image for the previous image
    min_period_previous=5,            # Min days before the reference image for the previous image
    same_sensor=False,                # Require both images from the same sensor
    partial_frequency=50,             # How often to save partial results
    publish_af=False,                 # If True, include `field_id` in the payload (registers result on the platform).
                                      # Default False — anonymous extraction, mirrors the rest of the processor family.
)


### 🔍 Test functions

In [ ]:
# Single-entity test data — note the required 'reference_date' field
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "OTHERS",
    "sowing_date": "2025-04-01",
    "reference_date": "2025-06-15",  # YYYY-MM-DD of the reference image
}

#### Test get_change_index_api

In [ ]:
print("\n--- Test: get_change_index_api ---")
try:
    result = extractor.get_change_index_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else str(result)[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")

#### Test get_change_index_api_safe

In [ ]:
print("\n--- Test: get_change_index_api_safe ---")
safe_result = extractor.get_change_index_api_safe(seasonfield_data)
print(safe_result)

#### Test format_change_index_json

In [ ]:
print("\n--- Test: format_change_index_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_change_index_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_change_index_json: No valid data from API.")

### 🔍 process_single_entity_change_index

In [ ]:
import pandas as pd

row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "OTHERS",
    "sowing_date": "2025-04-01",
    "reference_date": "2025-06-15",
})

result = extractor.process_single_entity_change_index(row)
print(result)

### 🔍 Missing-reference_date behavior

If a row has no `reference_date`, the processor returns a clean error (no retries) so bulk runs don't hammer the API:

In [ ]:
bad_row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "OTHERS",
})
print(extractor.process_single_entity_change_index(bad_row))

### 🔍 Column mapping — wire an upstream reference-date column

If your entity DataFrame carries the reference date under a different column (e.g. from an upstream `CoverageExtractor` that calls it `image_date`), map it to the canonical `reference_date` via `column_mapping`:

In [ ]:
upstream = pd.DataFrame([
    {
        "entity_id": "z361x33",
        "image_date": "2025-06-15",
        "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
        "crop": "OTHERS",
    }
])

extractor_mapped = ChangeIndexExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)
extractor_mapped.setup_change_index_parameters(
    map_type="NDVI",
    collections=["Sentinel-2"],
    column_mapping={
        "id": "entity_id",
        "reference_date": "image_date",
    },
)

result_bulk = extractor_mapped.process_change_index_bulk_extraction_parallel(
    entity_list=upstream,
    max_workers=2,
    output_path=manager.output_result_dir,
    prefix="change_index_mapped",
    skip_export=True,
)
print(f"Success: {result_bulk['successful_calculations']}/{result_bulk['total_calculations']}")
if not result_bulk['results_df'].empty:
    display(result_bulk['results_df'])

### 🔍 process_change_index_bulk_extraction_parallel

In [ ]:
top25 = manager.sfd_list.head(50).copy()
top25 = top25.rename(columns={"crop.id": "crop", "sowingDate": "sowing_date"})
# Stamp a reference_date column — in a real pipeline this comes from a coverage/emergence extractor.
top25["reference_date"] = "2025-06-15"
print(top25.columns)

result = extractor.process_change_index_bulk_extraction_parallel(
    entity_list=top25,
    params=None,
    max_workers=10,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop",
    filter_value="OTHERS",
    filter_type="include",
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")
print(result['results_df'].columns)
print("\n🔎 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
print(result["results_df"])

In [ ]:
results = result["results_df"]
print(results.columns)

In [ ]:
print(manager.output_result_dir)

## **publish_af parameter (align with the rest of the processor family)**

Like Baresoil / Covercrop / Emergence / Greenness / Harvest / Planted / score / InSeasonMonitoring,
ChangeIndex now exposes a `publish_af` flag that controls whether the entity's `field_id` is
included in the API payload:

- `publish_af=False` (default): payload carries only `crop`, `feature`, `sowing_date` —
  the API computes the change index without registering anything on the platform.
- `publish_af=True`: payload also carries `field_id`, so the result is associated with the
  entity on the platform side.

When `publish_af=True`, the entity row **must** have an `id`; with `publish_af=False`, the
id is only used for logging/result stamping and may be missing.

In [ ]:
# Inspect the payload directly via a `requests`-level patch — no API call needed.
import json
from unittest.mock import patch, MagicMock

demo_entity = {
    'id': 'z361x33',
    'geometry': 'POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.73031410000000, -58.93060599 -13.71867078, -58.94540508 -13.72028589))',
    'crop': 'OTHERS',
    'sowing_date': '2025-04-01',
    'reference_date': '2025-06-15',
}

def _capture_payload(extractor_, entity):
    """Run get_change_index_api against a mocked POST and return the JSON payload sent."""
    target = 'earthdaily.agriculture.processors.processor_change_index_functions.requests.post'
    with patch(target) as mock_post:
        mock_post.return_value = MagicMock(
            json=lambda: {'status': 'ok', 'data': {}},
            raise_for_status=lambda: None,
        )
        extractor_.get_change_index_api(entity)
        sent = json.loads(mock_post.call_args.kwargs['data'])
    return sent

# Default (publish_af=False) — payload omits field_id.
extractor.setup_change_index_parameters(publish_af=False)
payload_off = _capture_payload(extractor, demo_entity)
print('publish_af=False payload keys:', sorted(payload_off.keys()))
print('  field_id present:', 'field_id' in payload_off)

# publish_af=True — payload includes field_id.
extractor.setup_change_index_parameters(publish_af=True)
payload_on = _capture_payload(extractor, demo_entity)
print('publish_af=True  payload keys:', sorted(payload_on.keys()))
print('  field_id present:', 'field_id' in payload_on, '->', payload_on.get('field_id'))

# Reset to default so the rest of the notebook runs against the standard config.
extractor.setup_change_index_parameters(publish_af=False)

### Missing-id behaviour

With `publish_af=False`, an entity without an `id` is fine (entity_id falls back to `"unknown"`
for logging). With `publish_af=True`, a missing `id` is a hard error because the payload
would be incomplete.

In [ ]:
no_id_entity = {k: v for k, v in demo_entity.items() if k != 'id'}

extractor.setup_change_index_parameters(publish_af=False)
payload = _capture_payload(extractor, no_id_entity)
print('publish_af=False, no id   -> payload keys:', sorted(payload.keys()), '(no raise)')

extractor.setup_change_index_parameters(publish_af=True)
try:
    extractor.get_change_index_api(no_id_entity)
except ValueError as e:
    print('publish_af=True,  no id   -> ValueError:', e)

# Reset.
extractor.setup_change_index_parameters(publish_af=False)
